# Customer purchases — XGBoost
This notebook fits XGBoost to age and estimated salary, evaluates a stratified train/test split and saves the model and an HTML report. Run notebook 02 first, then run the cells here in order.

Recall is the training-CV objective because missed purchasers are the main concern. Precision, F1 and false-positive counts show the trade-off. `Purchased = 1` is the positive class.


## How this model works, feature choices and hyperparameters
XGBoost fits boosted trees sequentially to improve a regularized classification objective. Raw age/salary features do not need scaling or polynomial expansion. Median imputation is kept inside CV for a consistent treatment of incomplete rows; native missing-value handling is a possible separate experiment.

`objective="binary:logistic"` matches Purchased = 0/1. `eval_metric="logloss"` describes the internal binary fit; training-CV recall selects hyperparameters here. `n_estimators`, `learning_rate` and `max_depth` jointly control model complexity. `subsample=0.8` samples training rows; both available predictors remain eligible because `colsample_bytree=1.0`.

The test set is not an early-stopping eval_set. Tree counts are selected through training CV, then the chosen estimator fits only the training rows. Importance is not causal evidence, and the probability scale should be checked before probability-sensitive use.


## Setup and editable run settings


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split

locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations for candidate in (location, location / "classification")
     if (candidate / "customer_purchases_dataset/customer_details.csv").is_file()), None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root, classification/, or notebooks/.")
DATA_DIR = PROJECT_ROOT / "customer_purchases_dataset"
SOURCE_PATH = DATA_DIR / "customer_details.csv"
CLEANED_PATH = DATA_DIR / "cleaned_data.csv"
COMMON_ARTIFACTS = PROJECT_ROOT / "artifacts/common"
TARGET = "Purchased"
ID_COLUMN = "RowId"
FEATURE_COLUMNS = ["Age", "EstimatedSalary"]
CLASS_NAMES = ["Did not purchase", "Purchased"]
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
MODEL_SLUG = 'xgboost_classifier'
MODEL_TITLE = 'XGBoost'
MODEL_EXPLANATION = 'XGBoost fits boosted trees sequentially to improve a regularized classification objective. Raw age/salary features do not need scaling or polynomial expansion. Median imputation is kept inside CV for a consistent treatment of incomplete rows; native missing-value handling is a possible separate experiment.\n\n`objective="binary:logistic"` matches Purchased = 0/1. `eval_metric="logloss"` describes the internal binary fit; training-CV recall selects hyperparameters here. `n_estimators`, `learning_rate` and `max_depth` jointly control model complexity. `subsample=0.8` samples training rows; both available predictors remain eligible because `colsample_bytree=1.0`.\n\nThe test set is not an early-stopping eval_set. Tree counts are selected through training CV, then the chosen estimator fits only the training rows. Importance is not causal evidence, and the probability scale should be checked before probability-sensitive use.'
# Edit settings here. Each classifier controls and saves its own run.
config = {"random_state": 0, "test_size": 0.25, "cv_folds": 5,
          "tune": True, "scoring": "recall"}

if not 0 < config["test_size"] < 0.5:
    raise ValueError("test_size must be between 0 and 0.5")
if not isinstance(config["cv_folds"], int) or config["cv_folds"] < 2:
    raise ValueError("cv_folds must be an integer >= 2")
if not isinstance(config["tune"], bool):
    raise ValueError("tune must be boolean")
if config["scoring"] not in {"recall", "f1", "precision", "roc_auc", "average_precision", "balanced_accuracy", "accuracy"}:
    raise ValueError("Choose a supported classification scoring name")
display(pd.Series(config, name="run_settings").to_frame())


,run_settings
random_state,0
test_size,0.25
cv_folds,5
tune,True
scoring,recall


## Load the cleaned data and split
Age and estimated salary are the predictors. RowId is kept for tracing errors and Purchased is the target. The default stratified split gives 275 training rows and 92 test rows; its seed and test fraction match notebook 01.


In [3]:
required = [CLEANED_PATH, COMMON_ARTIFACTS / "cleaning_metadata.json"]
if any(not path.is_file() for path in required):
    raise FileNotFoundError("Run 02_data_cleaning.ipynb first to create the shared cleaned CSV.")
cleaning_metadata = json.loads((COMMON_ARTIFACTS / "cleaning_metadata.json").read_text())
if cleaning_metadata.get("format_version") != 1:
    raise ValueError("Unsupported cleaned-data metadata; rerun 02_data_cleaning.ipynb.")
if hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest() != cleaning_metadata["source_sha256"]:
    raise ValueError("Source CSV changed; rerun 02_data_cleaning.ipynb before training.")
if hashlib.sha256(CLEANED_PATH.read_bytes()).hexdigest() != cleaning_metadata["cleaned_sha256"]:
    raise ValueError("Cleaned CSV changed; rerun 02_data_cleaning.ipynb before training.")
data = pd.read_csv(CLEANED_PATH)
if list(cleaning_metadata["features"]) != FEATURE_COLUMNS or cleaning_metadata["target"] != TARGET:
    raise ValueError("Cleaned schema differs from this notebook; review notebook 02.")
if not data[ID_COLUMN].is_unique or set(data[TARGET].unique()) != {0, 1}:
    raise ValueError("Cleaned IDs or target labels are invalid; rerun notebook 02.")
training, testing = train_test_split(data, test_size=config["test_size"], random_state=config["random_state"], stratify=data[TARGET])
X_train, y_train = training[FEATURE_COLUMNS], training[TARGET]
X_test, y_test = testing[FEATURE_COLUMNS], testing[TARGET]
if y_train.value_counts().min() < config["cv_folds"]:
    raise ValueError("cv_folds exceeds the number of training rows in the smaller class")
assert not (set(training[ID_COLUMN]) & set(testing[ID_COLUMN]))
display(pd.DataFrame({"training": y_train.value_counts(), "testing": y_test.value_counts()}).sort_index())
print("Training / testing:", len(training), len(testing))


,training,testing
Purchased,,
0,172,57
1,103,35


Training / testing: 275 92


## Preprocessing and tuning
Imputation and any scaling or feature expansion are fitted within each training CV fold. Hyperparameters are selected by training CV, then the chosen estimator is fitted on the training split. The test set is used after fitting.


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
try:
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError("Install xgboost in this notebook's Python environment to run this model.") from error

preprocessing = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
estimator = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=1.0, objective="binary:logistic", eval_metric="logloss", random_state=config["random_state"], n_jobs=1)
model = Pipeline([*preprocessing, ("classifier", estimator)])
PARAM_GRID = {"classifier__n_estimators": [100, 250], "classifier__max_depth": [2, 3], "classifier__learning_rate": [0.05, 0.1]}
cv = StratifiedKFold(n_splits=config["cv_folds"], shuffle=True, random_state=config["random_state"])
if MODEL_SLUG == "knn_classifier":
    smallest_fold = min(len(indices) for indices, _ in cv.split(X_train, y_train))
    if estimator.n_neighbors > smallest_fold or max(PARAM_GRID["classifier__n_neighbors"]) > smallest_fold:
        raise ValueError("KNN neighbor counts exceed a CV training fold; reduce the grid or cv_folds")
if config["tune"]:
    search = GridSearchCV(model, PARAM_GRID, scoring=config["scoring"], cv=cv, n_jobs=1, error_score="raise", refit=True)
    search.fit(X_train, y_train)
    model = search.best_estimator_
    best_parameters = search.best_params_
    cv_score = float(search.best_score_)
    cv_results = pd.DataFrame(search.cv_results_)[["rank_test_score", "mean_test_score", "std_test_score", "params"]].sort_values("rank_test_score")
else:
    fold_scores = cross_val_score(model, X_train, y_train, scoring=config["scoring"], cv=cv, n_jobs=1, error_score="raise")
    model.fit(X_train, y_train)
    best_parameters = {}
    cv_score = float(fold_scores.mean())
    cv_results = pd.DataFrame({"fold": np.arange(1, len(fold_scores) + 1), "score": fold_scores})
print("Training CV objective:", config["scoring"], "score:", cv_score)
print("Fitted parameters:", model.named_steps["classifier"].get_params())
display(cv_results)


Training CV objective: recall score: 0.9123809523809523
Fitted parameters: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.05, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 2, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 100, 'n_jobs': 1, 'num_parallel_tree': None, 'random_state': 0, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}


,rank_test_score,mean_test_score,std_test_score,params
0,1,0.912381,0.070941,"{'classifier__learning_rate': 0.05, 'classifie..."
2,2,0.902381,0.081733,"{'classifier__learning_rate': 0.05, 'classifie..."
4,3,0.901905,0.088952,"{'classifier__learning_rate': 0.1, 'classifier..."
1,4,0.892381,0.085521,"{'classifier__learning_rate': 0.05, 'classifie..."
6,5,0.882381,0.098257,"{'classifier__learning_rate': 0.1, 'classifier..."
3,6,0.872857,0.093187,"{'classifier__learning_rate': 0.05, 'classifie..."
7,7,0.862857,0.086839,"{'classifier__learning_rate': 0.1, 'classifier..."
5,8,0.853333,0.084295,"{'classifier__learning_rate': 0.1, 'classifier..."


## Performance and errors
Accuracy, balanced accuracy, precision, recall and F1 use predicted classes. ROC AUC and average precision use positive-class probabilities. Undefined precision or recall is reported as zero when a class has no relevant predictions or observations.


In [5]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve, log_loss)

def continuous_scores(fitted_model, X):
    if hasattr(fitted_model, "predict_proba"):
        positive_index = list(fitted_model.classes_).index(1)
        return fitted_model.predict_proba(X)[:, positive_index], "purchase_probability"
    return fitted_model.decision_function(X), "decision_margin"

def evaluate_split(fitted_model, X, y, split_name):
    predicted = fitted_model.predict(X)
    scores, score_type = continuous_scores(fitted_model, X)
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite classification scores")
    metrics = {
        "dataset": split_name, "rows": len(y), "accuracy": accuracy_score(y, predicted),
        "balanced_accuracy": balanced_accuracy_score(y, predicted),
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted, zero_division=0),
        "f1": f1_score(y, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y, scores), "average_precision": average_precision_score(y, scores),
        "score_type": score_type,
    }
    if score_type == "purchase_probability":
        metrics["log_loss"] = log_loss(y, np.column_stack([1 - scores, scores]), labels=[0, 1])
    return metrics, predicted, scores, score_type

train_metrics, train_predictions, train_scores, score_type = evaluate_split(model, X_train, y_train, "Training")
test_metrics, test_predictions, test_scores, _ = evaluate_split(model, X_test, y_test, "Testing")
performance = pd.DataFrame([train_metrics, test_metrics])
confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
report_values = classification_report(y_test, test_predictions, labels=[0, 1],
                                      target_names=CLASS_NAMES, output_dict=True, zero_division=0)
class_report = pd.DataFrame({name: values for name, values in report_values.items() if isinstance(values, dict)}).T
predictions = testing[[ID_COLUMN, *FEATURE_COLUMNS, TARGET]].copy()
predictions["prediction"] = test_predictions
predictions[score_type] = test_scores
predictions["correct"] = predictions[TARGET] == predictions.prediction
predictions["error_type"] = np.select([
    (predictions[TARGET] == 1) & (predictions.prediction == 0),
    (predictions[TARGET] == 0) & (predictions.prediction == 1),
], ["False negative", "False positive"], default="Correct")
display(performance)
display(class_report)
display(predictions.loc[~predictions.correct])
print("Recall gap (training minus testing):", train_metrics["recall"] - test_metrics["recall"])
print("F1 gap (training minus testing):", train_metrics["f1"] - test_metrics["f1"])


,dataset,rows,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,score_type,log_loss
0,Training,275,0.930909,0.933083,0.881818,0.941748,0.910798,0.982953,0.968853,purchase_probability,0.180011
1,Testing,92,0.836957,0.846366,0.738095,0.885714,0.805195,0.912281,0.836437,purchase_probability,0.384010


,precision,recall,f1-score,support
Did not purchase,0.920000,0.807018,0.859813,57.0
Purchased,0.738095,0.885714,0.805195,35.0
macro avg,0.829048,0.846366,0.832504,92.0
weighted avg,0.850797,0.836957,0.839034,92.0


,RowId,Age,EstimatedSalary,Purchased,prediction,purchase_probability,correct,error_type
203,215,47.0,43000.0,0,1,0.775669,False,False positive
267,286,37.0,93000.0,1,0,0.327912,False,False negative
41,43,35.0,108000.0,0,1,0.751609,False,False positive
210,222,35.0,91000.0,1,0,0.150905,False,False negative
87,92,30.0,116000.0,0,1,0.662778,False,False positive
264,283,37.0,70000.0,1,0,0.033264,False,False negative
194,206,47.0,47000.0,0,1,0.775669,False,False positive
62,65,59.0,83000.0,0,1,0.938256,False,False positive
164,172,34.0,115000.0,0,1,0.729168,False,False positive
266,285,48.0,141000.0,0,1,0.947748,False,False positive


Recall gap (training minus testing): 0.05603328710124833
F1 gap (training minus testing): 0.1056033168709225


## Model-specific interpretation


In [6]:
interpretation = pd.DataFrame({"feature": FEATURE_COLUMNS, "split_importance": model.named_steps["classifier"].feature_importances_})
display(interpretation)


,feature,split_importance
0,Age,0.567344
1,EstimatedSalary,0.432656


## Decision regions
The boundary is learned from training rows and plotted in the original age and salary units. Held-out points show where the classifier makes errors.


In [7]:
# All figures describe this classifier only.
metric_long = performance.melt(id_vars="dataset", value_vars=["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"],
                               var_name="metric", value_name="score")
metric_figure = px.bar(metric_long, x="metric", y="score", color="dataset", barmode="group", title=f"{MODEL_TITLE}: train/test performance")
metric_figure.update_yaxes(range=[0, 1.05])
confusion_figure = px.imshow(confusion, x=CLASS_NAMES, y=CLASS_NAMES, text_auto=True,
                            color_continuous_scale="Blues", title="Test confusion matrix")
confusion_figure.update_layout(xaxis_title="Predicted class", yaxis_title="Actual class")
fpr, tpr, _ = roc_curve(y_test, test_scores)
roc_figure = go.Figure([
    go.Scatter(x=fpr, y=tpr, mode="lines", name=f"ROC AUC {test_metrics['roc_auc']:.3f}"),
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash"), name="No discrimination"),
])
roc_figure.update_layout(title="Test ROC: continuous model scores", xaxis_title="False positive rate", yaxis_title="True positive rate")
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_scores)
pr_figure = go.Figure([
    go.Scatter(x=recall_curve, y=precision_curve, mode="lines", name=f"Average precision {test_metrics['average_precision']:.3f}"),
    go.Scatter(x=[0, 1], y=[y_test.mean(), y_test.mean()], mode="lines", line=dict(dash="dash"), name="Test prevalence"),
])
pr_figure.update_layout(title="Test precision–recall curve", xaxis_title="Recall", yaxis_title="Precision")

# Use a 120×120 grid to keep the boundary plot manageable.
age_axis = np.linspace(data.Age.min() - 1, data.Age.max() + 1, 120)
salary_axis = np.linspace(max(0, data.EstimatedSalary.min() - 1000), data.EstimatedSalary.max() + 1000, 120)
age_grid, salary_grid = np.meshgrid(age_axis, salary_axis)
mesh = pd.DataFrame({"Age": age_grid.ravel(), "EstimatedSalary": salary_grid.ravel()})
boundary = model.predict(mesh).reshape(age_grid.shape)
boundary_figure = go.Figure(go.Contour(x=age_axis, y=salary_axis, z=boundary, contours=dict(start=0, end=1, size=1),
                                      colorscale=[[0, "#f6c8c8"], [1, "#c5e8ce"]], showscale=False, opacity=.7))
for label, name in enumerate(CLASS_NAMES):
    rows = testing.loc[testing[TARGET] == label]
    boundary_figure.add_trace(go.Scatter(x=rows.Age, y=rows.EstimatedSalary, mode="markers", name=name,
                                         marker=dict(color=["#c53030", "#237c43"][label], size=8),
                                         text=rows[ID_COLUMN], hovertemplate="Row %{text}<br>Age %{x}<br>Salary %{y}<extra></extra>"))
boundary_figure.update_layout(title="Decision regions in original units; held-out customers", xaxis_title="Age", yaxis_title="Estimated salary")
for figure in [metric_figure, confusion_figure, roc_figure, pr_figure, boundary_figure]:
    figure.show()


## Save the model and report
Rerunning overwrites this classifier's artifacts and report. The saved estimator includes preprocessing and accepts raw Age and EstimatedSalary columns.


In [8]:
from datetime import datetime, timezone
from html import escape
import joblib
import sklearn

MODEL_ARTIFACTS = PROJECT_ROOT / "artifacts" / MODEL_SLUG
REPORT_PATH = PROJECT_ROOT / "reports" / f"{MODEL_SLUG}.html"
MODEL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
training.to_csv(MODEL_ARTIFACTS / "training_data.csv", index=False)
testing.to_csv(MODEL_ARTIFACTS / "testing_data.csv", index=False)
performance.to_csv(MODEL_ARTIFACTS / "performance.csv", index=False)
predictions.to_csv(MODEL_ARTIFACTS / "test_predictions.csv", index=False)
class_report.rename_axis("class_or_summary").to_csv(MODEL_ARTIFACTS / "classification_report.csv")
pd.DataFrame(confusion, index=pd.Index(CLASS_NAMES, name="actual"), columns=CLASS_NAMES).to_csv(MODEL_ARTIFACTS / "confusion_matrix.csv")
cv_results.to_csv(MODEL_ARTIFACTS / "cv_results.csv", index=False)
interpretation.to_csv(MODEL_ARTIFACTS / "model_details.csv", index=False)
joblib.dump(model, MODEL_ARTIFACTS / "model.joblib")

def json_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return str(value)

metadata = {
    "format_version": 1, "model": MODEL_SLUG, "title": MODEL_TITLE,
    "created_at_utc": datetime.now(timezone.utc).isoformat(), "config": config,
    "source_sha256": cleaning_metadata["source_sha256"], "cleaned_sha256": cleaning_metadata["cleaned_sha256"],
    "features": FEATURE_COLUMNS, "target": TARGET, "positive_class": 1, "score_type": score_type,
    "split_rows": {"training": len(training), "testing": len(testing)},
    "training_row_ids": training[ID_COLUMN].tolist(), "testing_row_ids": testing[ID_COLUMN].tolist(),
    "best_parameters": best_parameters, "training_cv_score": cv_score,
    "fitted_estimator_parameters": model.named_steps["classifier"].get_params(),
    "preprocessing": [name for name, _ in model.steps[:-1]],
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
if MODEL_SLUG == "logistic_regression_classifier":
    metadata["polynomial_degree"] = POLYNOMIAL_DEGREE
(MODEL_ARTIFACTS / "run_metadata.json").write_text(json.dumps(metadata, indent=2, default=json_value), encoding="utf-8")

hyperparameters = pd.DataFrame({"parameter": list(model.named_steps["classifier"].get_params()),
                               "value": [str(value) for value in model.named_steps["classifier"].get_params().values()]})
notes = [
    "Purchased = 1 is the positive class. Recall measures captured purchasers; precision measures the purity of predicted purchasers.",
    f"Hyperparameters were {'selected with training-only CV' if config['tune'] else 'fixed in the notebook'}; the test set was not used for tuning.",
    "All learned feature/preprocessing steps are fitted inside CV folds and the final training split.",
    f"ROC and precision–recall use {score_type.replace('_', ' ')} scores; classification metrics use model.predict decisions.",
    "Train/test score differences are diagnostics; this small held-out sample does not guarantee future performance.",
    "This report describes one model family. No cross-model ranking or comparison is generated.",
]
body = [f"<h1>{escape(MODEL_TITLE)}: purchase classification</h1>",
        f"<p>Training rows: {len(training)}; test rows: {len(testing)}. Positive class: Purchased = 1.</p>",
        f"<p>Training CV objective: {escape(config['scoring'])}. {'Tuning enabled' if config['tune'] else 'Fixed hyperparameters'}.</p>",
        "<h2>Model and preprocessing notes</h2><p>" + escape(MODEL_EXPLANATION).replace("\n", "<br>") + "</p>"]
for heading, table in [("Training and testing metrics", performance), ("Per-class test results", class_report.reset_index()),
                       ("Fitted hyperparameters", hyperparameters), ("Training CV results", cv_results),
                       ("Model details", interpretation.reset_index()), ("Misclassified held-out customers", predictions.loc[~predictions.correct])]:
    body.append(f"<h2>{escape(heading)}</h2>" + table.to_html(index=False, border=0, float_format=lambda value: f"{value:.4f}"))
for i, (heading, figure) in enumerate(zip(
    ["Train/test scores", "Confusion matrix", "ROC", "Precision–recall", "Decision regions"],
    [metric_figure, confusion_figure, roc_figure, pr_figure, boundary_figure],
)):
    body.append(f"<h2>{heading}</h2>" + pio.to_html(figure, full_html=False, include_plotlyjs=i == 0))
body.append("<h2>Interpretation notes</h2><ul>" + "".join(f"<li>{escape(note)}</li>" for note in notes) + "</ul>")
report_html = "<!doctype html><html><head><meta charset='utf-8'><title>" + escape(MODEL_TITLE) + "</title>" +     "<style>body{font-family:Arial,sans-serif;max-width:1200px;margin:2rem auto;padding:0 1rem;color:#243447}h2{margin-top:2rem}table{border-collapse:collapse;display:block;overflow-x:auto}td,th{border:1px solid #ddd;padding:7px;text-align:right}th{background:#eaf2f8}</style></head><body>" + "".join(body) + "</body></html>"
REPORT_PATH.write_text(report_html, encoding="utf-8")
print("Saved artifacts:", MODEL_ARTIFACTS.relative_to(PROJECT_ROOT))
print("Saved report:", REPORT_PATH.relative_to(PROJECT_ROOT))
display(HTML(f'<a href="../reports/{MODEL_SLUG}.html" target="_blank">Open {escape(MODEL_TITLE)} report</a>'))


Saved artifacts: artifacts/xgboost_classifier
Saved report: reports/xgboost_classifier.html


## Results and next step
On the saved 92-row test split, the model identifies 31 of 35 purchasers, misses 4, and incorrectly flags 11 non-purchasers. Test recall is 88.6%, precision 73.8% and F1 0.805.

Training recall is 94.2% and precision 88.2%. Training CV recall is 0.912; it was used to choose parameters, so the held-out result is the more useful check of this fit.

CV selected 100 trees, depth 2 and a learning rate of 0.05. Test recall and precision are lower than training scores. A small comparison of shallower trees and stronger regularization on training CV would check whether the fit can be made more stable. Any further tuning should use training folds rather than repeated checks of this test split.
